# Step 4 — Sentence embeddings (Stage A: pretrained, no training)

**Input:** `outputs/equal_step2_masking/step2_masked_analysis_text.csv` (cased, with line breaks and bullet markers still present).
**Output:** sentence units and their SentenceTransformers embeddings, plus an exploratory comparison with a TF-IDF baseline.

Sentences are segmented from the Step 2 masked text rather than the Step 3 text, because Step 3 collapses the line breaks and bullet markers that mark sentence boundaries, and lowercasing is unnecessary for the embedding model. Step 3 text remains for lexical methods.

No model is trained here. Clusters are exploratory and are not bias judgments.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

INPUT_PATH = Path("outputs") / "equal_step2_masking" / "step2_masked_analysis_text.csv"
OUTPUT_DIR = Path("outputs") / "equal_step4_sentence_embeddings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Multilingual because the advertisements may contain Filipino/Taglish wording.
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
N_CLUSTERS = 6
RANDOM_STATE = 42

data = pd.read_csv(INPUT_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"{len(data):,} advertisements loaded.")

D:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1,500 advertisements loaded.


## 1. Segmentation

Each text component is split by line; leading bullet markers are removed; lines are split further at `.`, `!` or `?` followed by whitespace. Units without any letter are dropped. Case and punctuation are kept. The job title is kept as its own unit (`component = job_title`).

In [2]:
# Canonical units come from Step 2 (section 7), which protects abbreviations such as Ms., Sr. and B.S.
UNITS_PATH = Path("outputs") / "equal_step2_masking" / "step2_masked_sentence_units.csv"
units_step2 = pd.read_csv(UNITS_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
assert units_step2["sentence_id"].is_unique
assert set(units_step2["record_id"]).issubset(set(data["record_id"]))
sentences = units_step2[["sentence_id", "record_id", "component", "sentence"]].merge(
    data[["record_id", "employer_id"]], on="record_id", how="left", validate="many_to_one"
)[["sentence_id", "record_id", "employer_id", "component", "sentence"]]
print(f"{len(sentences):,} sentence units from {sentences['record_id'].nunique():,} advertisements.")
print(sentences["component"].value_counts().to_string())
print(f"Exact-duplicate sentence texts: {sentences['sentence'].duplicated().sum():,}")
display(sentences.sample(8, random_state=RANDOM_STATE))

7,971 sentence units from 1,500 advertisements.
component
qualification    6374
job_title        1497
body_other        100
Exact-duplicate sentence texts: 3,255


,sentence_id,record_id,employer_id,component,sentence
2716,row-0633-s003,row-0633,EMP-957f35e1,qualification,Bachelor degree of Human Resource or related courses
7608,row-1436-s003,row-1436,EMP-a3523ee6,qualification,"At least College Graduate, with Below 1 Year Exp"
5251,row-1022-s001,row-1022,EMP-809c2dd4,job_title,Kowa Analyst Trainee
7338,row-1388-s005,row-1388,EMP-87cd9e75,qualification,"Willing to be assigned at LISP 1, Cabuyao (First Sumiden)"
6446,row-1230-s005,row-1230,EMP-1218889e,qualification,Effective communication and interpersonal skills
5514,row-1067-s002,row-1067,EMP-70f4a777,qualification,Graduate of BS Accountancy or BS in Accounting Technology
2287,row-0544-s001,row-0544,EMP-bbf628b3,job_title,Production Section Supervisor
736,row-0145-s003,row-0145,EMP-299e55a5,qualification,At least 3 years' relevant experience performing the same role in a construction setting


## 2. Embeddings

Identical sentence strings are encoded once and mapped back, so repeated template lines receive identical vectors. Vectors are L2-normalized, so dot product equals cosine similarity.

In [3]:
model = SentenceTransformer(MODEL_NAME)
unique_text, inverse = np.unique(sentences["sentence"].to_numpy(), return_inverse=True)
unique_vectors = model.encode(
    unique_text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True
)
embeddings = unique_vectors[inverse].astype("float32")
assert embeddings.shape == (len(sentences), model.get_sentence_embedding_dimension())
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4)
print(f"Model: {MODEL_NAME}; embedding matrix: {embeddings.shape}; unique texts encoded: {len(unique_text):,}")

np.save(OUTPUT_DIR / "step4_sentence_embeddings.npy", embeddings)
sentences.to_csv(OUTPUT_DIR / "step4_sentence_units.csv", index=False, encoding="utf-8-sig")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2265.64it/s]

Model: paraphrase-multilingual-MiniLM-L12-v2; embedding matrix: (7971, 384); unique texts encoded: 4,716


C:\Users\cyber\AppData\Local\Temp\ipykernel_35740\3153344227.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert embeddings.shape == (len(sentences), model.get_sentence_embedding_dimension())


## 3. Sanity check: nearest neighbours

For a few sentences containing occupational or gendered wording, the most similar other (distinct) sentences are shown. This is a qualitative check that the space reflects meaning.

In [4]:
probe_pattern = r"\b(?:female|male|barista|mechanic|salesman|saleswoman|young|aggressive|strong|supportive)\b"
probes = sentences[sentences["sentence"].str.contains(probe_pattern, case=False, regex=True)]
probes = probes.drop_duplicates("sentence").sample(min(4, len(probes)), random_state=RANDOM_STATE)
for idx in probes.index:
    scores = embeddings @ embeddings[idx]
    order = [j for j in np.argsort(-scores) if sentences.at[j, "sentence"] != sentences.at[idx, "sentence"]][:3]
    print(f"\nQUERY: {sentences.at[idx, 'sentence']}")
    for j in order:
        print(f"  {scores[j]:.3f}  {sentences.at[j, 'sentence']}")


QUERY: Strong computer navigation skills and PC Knowledge
  0.790  Sufficient knowledge in Computer Navigation, typing etc.
  0.730  Advanced computer skills
  0.730  Advanced computer skills



QUERY: Female or Male
  0.997  Male or female
  0.997  Male or Female
  0.997  Male or Female



QUERY: With proven experience in driving and mechanic for all kinds of vehicle under license class and related field
  0.793  With driving skill & license
  0.793  With driving skill & license
  0.745  Preferably with TESDA Certification in driving and/or vocational in training automotive/engine mechanics



QUERY: Preferably male
  0.995  Preferably Male
  0.995  Preferably Male
  0.995  Preferably Male


### Example cosine similarities

Cosine similarity between selected sentence pairs, read from the saved embeddings (the dot product of L2-normalised vectors). The sentences are taken from the dataset and are not edited.

In [5]:
EXAMPLE_PAIRS = [
    ("Strong mathematical and analytical skills", "With strong data-gathering and analytical skills"),
    ("Strong mathematical and analytical skills", "Above average interpersonal skills"),
    ("Strong mathematical and analytical skills", "Reliable and trustworthy"),
    ("Strong mathematical and analytical skills", "Ability to prioritize and lead multiple projects simultaneously"),
    ("Above average interpersonal skills", "With good oral and written communication and interpersonal skills"),
    ("Male", "Female"),
    ("Female", "Male/Female"),
]
first_index = sentences.drop_duplicates("sentence").set_index("sentence").index
position = {text: sentences.index[sentences["sentence"] == text][0] for text in {t for pair in EXAMPLE_PAIRS for t in pair}}
example_cosines = pd.DataFrame(
    [
        {"sentence_a": a, "sentence_b": b, "cosine": float(embeddings[position[a]] @ embeddings[position[b]])}
        for a, b in EXAMPLE_PAIRS
    ]
).round({"cosine": 3})
display(example_cosines)

,sentence_a,sentence_b,cosine
0,Strong mathematical and analytical skills,With strong data-gathering and analytical skills,0.705
1,Strong mathematical and analytical skills,Above average interpersonal skills,0.467
2,Strong mathematical and analytical skills,Reliable and trustworthy,0.339
3,Strong mathematical and analytical skills,Ability to prioritize and lead multiple projects simultaneously,0.260
4,Above average interpersonal skills,With good oral and written communication and interpersonal skills,0.608
5,Male,Female,0.454
6,Female,Male/Female,0.682


### Filipino/Taglish wording in the advertisements

A sentence is flagged when it contains at least one common Filipino function word from a fixed marker list. Ambiguous words that are also English or Spanish (for example `sa`, `para`, `na`) are left out, so the counts are a conservative lower bound, not a full language detection.

In [6]:
FILIPINO_MARKERS = ["ang", "mga", "nais", "po", "maaari", "aplikante", "ng", "lang"]
filipino_re = re.compile(r"\b(?:" + "|".join(FILIPINO_MARKERS) + r")\b", re.I)
sentences["has_filipino_marker"] = sentences["sentence"].str.contains(filipino_re)
flagged = sentences[sentences["has_filipino_marker"]]
print(
    f"{len(flagged):,} of {len(sentences):,} units ({len(flagged) / len(sentences):.2%}) contain Filipino/Taglish markers, "
    f"from {flagged['record_id'].nunique():,} of {sentences['record_id'].nunique():,} advertisements "
    f"({flagged['record_id'].nunique() / sentences['record_id'].nunique():.2%})."
)
display(flagged[["sentence_id", "component", "sentence"]].drop_duplicates("sentence").head(10))

47 of 7,971 units (0.59%) contain Filipino/Taglish markers, from 44 of 1,500 advertisements (2.93%).


,sentence_id,component,sentence
245,row-0050-s007,qualification,"Para po sa mga interesadong aplikante, maaari po kayong magpasa ng inyong updated resume sa [EMPLOYER] Boac at hanapin lamang po si Ms. ..."
246,row-0050-s008,qualification,Atin pong sundin ang minimum health protocol
253,row-0051-s007,qualification,"Para po sa mga interesadong aplikante, maaari po kayong magpasa ng inyong updated resume sa [EMPLOYER] Boac at hanapin lamang po si Sir."
1009,row-0286-s007,qualification,"Para po sa mga nais mag-apply, tingnan ang mga qualifications at i-click ang link para sagutan ang online application form."
2372,row-0563-s009,qualification,Iclick ang link para para makita ang buong detalye sa proseso ng paglahok
7070,row-1339-s009,qualification,"Sa mga nais mag-apply, i-click lang ang link sa baba o i-scan ang QR code para makapag register at antayin ang text para sa schedule."


## 4. Exploratory clustering and TF-IDF comparison

K-means with the same number of clusters is run on the embeddings and on TF-IDF features. The adjusted Rand index (ARI) measures how much the two partitions agree; low agreement is expected because they capture different structure. Silhouette values are computed on a random subsample and are not comparable across different feature spaces; they are shown only as a rough indication of separation.

In [7]:
emb_km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(embeddings)
sentences["embedding_cluster"] = emb_km.labels_

tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=3, max_df=0.5, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(sentences["sentence"])
tfidf_km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(tfidf_matrix)
sentences["tfidf_cluster"] = tfidf_km.labels_

rng = np.random.default_rng(RANDOM_STATE)
sub = rng.choice(len(sentences), size=min(5000, len(sentences)), replace=False)
print(f"ARI (embedding vs TF-IDF clusters): {adjusted_rand_score(sentences['embedding_cluster'], sentences['tfidf_cluster']):.3f}")
print(f"Silhouette, embeddings (cosine, n={len(sub):,}): {silhouette_score(embeddings[sub], emb_km.labels_[sub], metric='cosine'):.3f}")
print(f"Silhouette, TF-IDF (cosine, n={len(sub):,}): {silhouette_score(tfidf_matrix[sub], tfidf_km.labels_[sub], metric='cosine'):.3f}")

display(pd.crosstab(sentences["embedding_cluster"], sentences["component"], margins=True))
display(pd.crosstab(sentences["embedding_cluster"], sentences["tfidf_cluster"], margins=True))

ARI (embedding vs TF-IDF clusters): 0.228


Silhouette, embeddings (cosine, n=5,000): 0.156


Silhouette, TF-IDF (cosine, n=5,000): 0.059


component,body_other,job_title,qualification,All
embedding_cluster,,,,
0,35,14,1524,1573
1,5,8,1501,1514
2,15,7,1176,1198
3,24,45,907,976
4,14,2,580,596
5,7,1421,686,2114
All,100,1497,6374,7971


tfidf_cluster,0,1,2,3,4,5,All
embedding_cluster,,,,,,,
0,1543,0,28,0,0,2,1573
1,582,1,929,0,0,2,1514
2,239,727,17,0,0,215,1198
3,617,4,351,0,0,4,976
4,178,0,18,193,207,0,596
5,2076,0,37,0,0,1,2114
All,5235,732,1380,193,207,224,7971


In [8]:
# Representative sentences: the units closest to each embedding-cluster centroid.
summary_rows = []
for cluster in range(N_CLUSTERS):
    members = np.flatnonzero(emb_km.labels_ == cluster)
    closest = members[np.argsort(-(embeddings[members] @ emb_km.cluster_centers_[cluster]))]
    reps = sentences.loc[closest, "sentence"].drop_duplicates().head(5).tolist()
    summary_rows.append(
        {
            "embedding_cluster": cluster,
            "units": len(members),
            "advertisements": sentences.loc[members, "record_id"].nunique(),
            "employers": sentences.loc[members, "employer_id"].nunique(),
            "representative_sentences": " || ".join(reps),
        }
    )
cluster_summary = pd.DataFrame(summary_rows)
for row in summary_rows:
    print(f"\nCluster {row['embedding_cluster']}: {row['units']:,} units, {row['advertisements']:,} ads, {row['employers']:,} employers")
    for text in row["representative_sentences"].split(" || "):
        print(f"   - {text}")

sentences.to_csv(OUTPUT_DIR / "step4_sentence_units.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(OUTPUT_DIR / "step4_embedding_cluster_summary.csv", index=False, encoding="utf-8-sig")
print("\nSaved:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")


Cluster 0: 1,573 units, 680 ads, 205 employers
   - With good analytical, technical, communication and writing skills
   - With good communication skills and initiative
   - With good organizational and communication skills
   - With convincing communication skills
   - Strong analytical and good communication skills

Cluster 1: 1,514 units, 1,007 ads, 267 employers
   - With at least 1 year related working experience
   - With at least 1 year work experience
   - With 1-3 years working experience
   - With 1-2 years of working experience
   - At least 1 year working experience

Cluster 2: 1,198 units, 1,086 ads, 275 employers
   - Graduate of any degree
   - College Level/ Vocational Graduate
   - Graduate of any course
   - Graduate of any Bachelor degree
   - At least College/Vocational Graduate

Cluster 3: 976 units, 671 ads, 206 employers
   - Willing to be assigned in Panacan, Davao City
   - Willing to be assigned in Bajada, Davao City
   - Willing to be assigned in Catalunan P

## Limitations

- The pretrained model carries its own social biases, and embedding similarity is not a bias judgment.
- Repeated employer template lines inflate some clusters; cluster-level employer counts above show how concentrated each cluster is.
- Sentence units are analysis segments, not independent observations; sentences from the same advertisement or employer are correlated.
- K-means with a fixed number of clusters is a convenience, not evidence of six natural groups.